
# The `Vacuum` class

`jaxvacua.vacuum` is the package's **record layer**: the type you hand to storage,
compare against another solve, and ask "is this actually a physical vacuum?".
Everything upstream of it — sampling, Newton refinement, coni-LCS expansions —
produces loose arrays; this is where those arrays become an object that carries its
own meaning and can validate itself.

This notebook is a complete tour, built on two vacua taken from the other
tutorials so nothing has to be re-derived here:

| | geometry | source |
|---|---|---|
| **LCS vacuum** | $h^{1,2} = 2$, `model_ID=1` (the degree-18 hypersurface) | the SUSY solution of [NB05 — Finding flux vacua](../02_vacuum_finding/05_finding_flux_vacua.ipynb) |
| **coni-LCS vacuum** | $h^{1,2} = 3$, CYTools polytope, `conifold_basis=True` | vacuum #1 of [NB10 — coni-LCS pipeline](../03_geometry_and_limits/10_coniLCS_pipeline.ipynb), i.e. of [arXiv:2009.03312](https://arxiv.org/abs/2009.03312) |

The pair is deliberate: the class behaves *differently* in the two limits, and the
contrast is the clearest way to see what `limit` controls.

## Outline

1. [Setup](#setup)
2. [What a `Vacuum` is](#1-what-a-vacuum-is) — flux + location, and the two coordinate views
3. [What the constructor refuses](#2-what-the-constructor-refuses) — the invariants you can rely on
4. [`limit` and the conifold gate](#3-limit-and-the-conifold-gate) — the LCS/coni-LCS contrast
5. [Diagnostics: the structured report](#4-diagnostics-the-structured-report) — `diagnostics` and `is_consistent`
6. [Skipped is not passed](#5-skipped-is-not-passed)
7. [Broken vacua name their own failure](#6-broken-vacua-name-their-own-failure)
8. [The derived tier: `VacuumAnalysis`](#7-the-derived-tier-vacuumanalysis) — cached eigenvalues and stability
9. [The alignment scalar](#8-the-alignment-scalar) — conifold-only
10. [The `PFV` subclass](#9-the-pfv-subclass) — algebra data, racetrack estimate, PFV conditions
11. [Three notions of sameness](#10-three-notions-of-sameness) — exact, tolerant, up-to-duality
12. [Deduplicating an ensemble](#11-deduplicating-an-ensemble)
13. [Storage: three tiers](#12-storage-three-tiers) — and the subclass registry
14. [Feature reference](#13-feature-reference)
15. [Take-aways](#take-aways)
16. [Further reading](#further-reading)

## Setup


In [ ]:

import warnings
import os
import tempfile

import numpy as np
from scipy.optimize import root

import jax
import jax.numpy as jnp
jax.config.update("jax_enable_x64", True)

import jaxvacua as jvc
from jaxvacua.vacuum import (
    Vacuum, PFV, PFVData, VacuumAnalysis,
    real_to_complex, complex_to_real,
    save_vacua, load_vacua, vacuum_to_json, vacuum_from_json,
    unique_vacua,
)

warnings.filterwarnings("ignore")
print(f"jaxvacua {jvc.__version__}")


## 1. What a `Vacuum` is

A `Vacuum` is defined by exactly two things:

* its **flux** — the integer vector $\vec\Phi$ of length $2n_{\rm fluxes}$, and
* its **location** in moduli space.

The location has two equivalent descriptions, and you may supply either:

$$
\underbrace{(\vec z,\ \tau)}_{\text{complex}}
\qquad\Longleftrightarrow\qquad
\underbrace{x = [\operatorname{Re}z_1,\ \operatorname{Im}z_1,\ \dots,\ \operatorname{Re}\tau,\ \operatorname{Im}\tau]}_{\text{interleaved real vector the solvers consume}}
$$

Whichever you omit is derived in the constructor, so **both are always populated
afterwards**. `x` is the canonical stored form.

Everything else — $W_0$, $DW$, the residual, $g_s$ — is a *solved diagnostic*: a
number the producing solver happened to compute, defaulting to `NaN` when it did not.

We start from the SUSY solution of NB05. Note the residual: those printed values are
the ones NB05 displays, and they are **not** converged to the default tolerance —
section 4 will show the class telling us so, and we fix it there.

In [ ]:

# ---- the LCS geometry and the solution from NB05 -------------------------
# A FluxVacuaFinder (rather than a bare FluxEFT) because it carries a *sampler*,
# whose weak-coupling floor one of the checks in section 5 needs.
finder = jvc.FluxVacuaFinder(h12=2, model_ID=1)
print(f"h12 = {finder.h12}    D3 tadpole cap Q = {finder.Q()}")

# NB05, "SUSY solution" section -- copied verbatim
z_SUSY = jnp.array([0.3168651582886485 + 3.2301486028589927j,
                    0.37113099110064307 + 3.0746240438915935j])
tau_SUSY = -0.29834960566944685 + 6.803576151333438j
flux_SUSY = jnp.array([4., -3., -2., -2., 3., 2., 39., -13., -4., 0., -0., 0.])

In [ ]:

def as_vacuum(model, z, tau, flux, limit="LCS", **extra):
    """Package a solved point as a Vacuum, filling the solved diagnostics."""
    DW = model.DW(z, jnp.conj(z), tau, jnp.conj(tau), flux)
    W0 = model.superpotential(z, tau, flux, normalise=True)   # sqrt(2/pi) convention
    return Vacuum(
        flux=np.asarray(flux),                   # required
        z=np.asarray(z), tau=complex(tau),       # ... and so is a location
        limit=limit,
        DW=np.asarray(DW),
        residual=float(jnp.max(jnp.abs(DW))),
        W0=complex(W0),
        gs=float(1.0 / jnp.imag(tau)),
        **extra,
    )


vac_lcs_raw = as_vacuum(finder, z_SUSY, tau_SUSY, flux_SUSY)
vac_lcs_raw.short_summary()


### The two coordinate views

`x` was assembled from `(z, tau)`; both are now available and consistent. The layout
is stated once, model-free, by `real_to_complex` / `complex_to_real` — the same
convention the jitted `FluxEFT._convert_real_to_complex` implements, so you can move
between the two without a model in hand.

In [ ]:

print(f"z   = {vac_lcs_raw.z}")
print(f"tau = {vac_lcs_raw.tau}")
print(f"x   = {np.asarray(vac_lcs_raw.x)}")
print()

# the helpers are exact inverses, and need no model
z_rt, tau_rt = real_to_complex(complex_to_real(vac_lcs_raw.z, vac_lcs_raw.tau))
print(f"round-trip exact: z {np.array_equal(z_rt, vac_lcs_raw.z)}, "
      f"tau {tau_rt == vac_lcs_raw.tau}")

# ... and agree with the model-bound, jitted version
z_m, _, tau_m, _ = finder._convert_real_to_complex(vac_lcs_raw.x)
print(f"agrees with FluxEFT._convert_real_to_complex: "
      f"{np.allclose(np.asarray(z_m), vac_lcs_raw.z, rtol=0, atol=0)}")


## 2. What the constructor refuses

The invariants are enforced at construction, not left for a later `None` check. This
matters because a half-built record propagates: it reaches storage, gets reloaded, and
fails somewhere far from where it was created.

In [ ]:

x_ok = np.asarray(vac_lcs_raw.x)
f_ok = np.asarray(flux_SUSY)

attempts = [
    ("no flux at all",             lambda: Vacuum(x=x_ok)),
    ("flux=None explicitly",       lambda: Vacuum(flux=None, x=x_ok)),
    ("flux but no location",       lambda: Vacuum(flux=f_ok)),
    ("z without tau",              lambda: Vacuum(flux=f_ok, z=np.array([1 + 1j]))),
    ("x and (z, tau) disagreeing", lambda: Vacuum(flux=f_ok, x=x_ok,
                                                 z=np.array([9 + 9j]),
                                                 tau=complex(0, 6.8))),
    ("x of odd length",            lambda: Vacuum(flux=f_ok, x=np.array([1.0, 2.0, 3.0]))),
]

for label, build in attempts:
    try:
        build()
        print(f"  {label:28s} -> accepted (!)")
    except (TypeError, ValueError) as exc:
        print(f"  {label:28s} -> {type(exc).__name__}: {str(exc).splitlines()[0][:72]}")


Two of those deserve a comment.

**`x` and `(z, tau)` disagreeing raises** rather than silently preferring one. If the
class picked a winner, a caller who passed both by mistake would never learn which of
their two descriptions was used — and the two describe different physics.

**A vacuum with no location is not a vacuum.** This is why
`PFV.from_quantum_numbers` now raises when the $N$-matrix is singular (section 9):
without a flat direction there is no point to sit at.


## 3. `limit` and the conifold gate

Near a conifold point the moduli space has a distinguished direction: the shrinking
cycle's modulus $z_{\rm cf}$, whose period is logarithmic. A vacuum there carries
three extra quantities — $z_{\rm cf}$ itself and the residual split into its conifold
and bulk parts.

**For a plain LCS vacuum those are not unknown, they are undefined.** There is no
conifold direction to split along. So the class gates them on `limit`:

```python
@property
def has_conifold(self) -> bool:
    return str(self.limit) in ("coniLCS", "coniLCS_series", "coniLCS_bulk")
```

and when it is `False` the three fields are `None` — *not* `NaN`. That choice is
load-bearing: `NaN != NaN`, so a `NaN` placeholder makes two otherwise identical
records compare unequal, which has already caused a real bug in the promotion layer.

Let us build the coni-LCS counterpart. The geometry and the flux quanta are NB10's
vacuum #1; we re-solve $DW = 0$ from the PFV locus, exactly as NB10 does.

In [ ]:

from cytools import Polytope

poly = Polytope([[-1, 3, -2, -1], [1, -1, 0, 0], [-1, 0, 0, 1], [-1, 0, 0, 0],
                 [-1, 0, 1, 1], [-1, 0, 2, 0], [-1, 0, 1, 0]])
cy = poly.triangulate().get_cy()

basis_matrix = np.array([[0, 1, 1], [1, 1, 0], [0, 0, 1]])
conifold_curve = np.array([-1, 1, 0])

model_coni = jvc.FluxVacuaFinder(
    h12=cy.h11(), Q=cy.h11() + cy.h12() + 2,
    use_cytools=True, mirror_cy=cy, ncf=2, use_gvs=True,
    maximum_degree=6, basis_change=basis_matrix,
    conifold_curve=conifold_curve, limit="coniLCS",
    prange=100, conifold_basis=True,
)
print(f"h12 = {model_coni.h12}   Q = {model_coni.Q()}   "
      f"limit = {model_coni.periods.limit}")

In [ ]:

# NB10 vacuum #1 (arXiv:2009.03312): seed on the PFV locus, then solve DW = 0.
M_coni = np.array([4, -8, 8])
K_coni = np.array([-8, 3, -6])
flux_coni = model_coni.pfv_to_flux(M_coni, K_coni)

tau_seed = 1j / 0.38
z_seed = model_coni.pfv_to_moduli(M_coni, K_coni, tau_seed)
x_seed = model_coni._convert_complex_to_real(z_seed, jnp.conj(z_seed),
                                             tau_seed, jnp.conj(tau_seed))

sol = root(model_coni.DW_x, x_seed, args=(flux_coni,),
           jac=model_coni.dDW_x, tol=1e-10, method="hybr")
x_coni = sol.x
z_coni, _, tau_coni, _ = model_coni._convert_real_to_complex(x_coni)
DW_coni = np.asarray(model_coni.DW_x(x_coni, flux_coni))
print(f"converged: {sol.success}    max|DW| = {np.max(np.abs(DW_coni)):.3e}")

In [ ]:

# The conifold direction sits in slot 0 (that is what `conifold_basis=True` and the
# basis change arrange), so DW[:2] are its real/imag parts and DW[2:] the bulk.
vac_coni = Vacuum(
    flux=np.asarray(flux_coni),
    x=np.asarray(x_coni),
    limit="coniLCS",                                     # <- opens the gate
    zcf=complex(z_coni[0]),
    residual=float(np.max(np.abs(DW_coni))),
    residual_conifold=float(np.max(np.abs(DW_coni[:2]))),
    residual_bulk=float(np.max(np.abs(DW_coni[2:]))),
    W0=complex(model_coni.superpotential(z_coni, tau_coni, flux_coni, normalise=True)),
    gs=float(1.0 / jnp.imag(tau_coni)),
)
vac_coni.short_summary()


### The same class, two behaviours

Put the summaries side by side. The LCS record shows no $z_{\rm cf}$ and no split —
not a row of `nan`s where the conifold data would be, but no row at all.

In [ ]:

for name, v in (("LCS", vac_lcs_raw), ("coni-LCS", vac_coni)):
    print(f"===== {name}  (has_conifold = {v.has_conifold}) "
          f"{'=' * (44 - len(name))}")
    v.long_summary()
    print()

print("the gated fields, read directly:")
for name, v in (("LCS", vac_lcs_raw), ("coni-LCS", vac_coni)):
    print(f"  {name:9s} zcf={v.zcf}  residual_bulk={v.residual_bulk}  "
          f"residual_conifold={v.residual_conifold}")


Reading a gated field when it does not apply returns `None` rather than raising, so
`to_dict` stays total and generic code can walk every field without special cases.

Two physics observations from the coni-LCS record, both visible in the summary above:

* $|W_0| \sim 10^{-4}$ against $|W_0| \sim 10^{2}$ for the generic LCS vacuum. The
  exponentially small superpotential is the whole point of the PFV construction; a
  generic flux vacuum has $|W_0| = \mathcal{O}(10^2)$ in this normalisation.
* the residual **split** is informative on its own: the bulk directions are solved to
  $\sim 10^{-15}$ while the conifold direction sits at $\sim 10^{-13}$. The steep
  logarithmic period makes that direction numerically the hardest, which is exactly
  why the split is recorded separately rather than only the maximum.


## 4. Diagnostics: the structured report

`is_physical` in `jaxvacua.flux_utils` answers "is this point physical?" with a bare
`bool`, and tells you *which* condition failed only by printing when
`verbose=True`. A caller cannot act on that, which is why several partial
re-implementations of the cascade grew across the workspace.

The class replaces it with a pair:

```python
vac.diagnostics(model)     # -> {check_name: (ok, value, reason)}
vac.is_consistent(model)   # -> bool, a reduction of exactly that report
```

The boolean **is** the reduction, so the verdict and the explanation cannot disagree.
The model is a required argument — every non-trivial check needs the geometry, and an
optional model would silently degrade to a near-empty report.

### The checks

| key | tests | needs | cost |
|---|---|---|---|
| `residual` | $\max\|DW\| <$ `residual_tol` | — | $O(1)$ |
| `im_tau_positive` | $\operatorname{Im}\tau > 0$ | — | $O(1)$ |
| `flux_integrality` | $\|f - \mathrm{round}(f)\| < 10^{-9}$ | — | $O(1)$ |
| `runaway_bound` | $\max(\|z\|, \|\tau\|) \le$ `moduli_max` | kwarg | $O(1)$ |
| `dilaton_floor` | $\operatorname{Im}\tau >$ `s_min` | sampler / kwarg | $O(1)$ |
| `kahler_cone` | $H \cdot \operatorname{Im} z > 0$ | hyperplanes | $O(h_{12}^2)$ |
| `tadpole` | **signed** $0 < f\cdot\Sigma\cdot h \le Q$ | `model.Q()` | $O(h_{12}^2)$ |
| `kahler_metric_pd` | $\mathrm{eig}(K) > 0$ | `model.kahler_metric` | $O(h_{12}^3)$ |
| `hessian_min_eig` | opt-in, `stability=True` | Hessian | expensive |

Three of those encode decisions worth stating explicitly.

**$\operatorname{Im} z > 0$ is deliberately *not* a check.** Only
$\operatorname{Im}\tau > 0$ is a physicality requirement; the moduli are constrained
by Kähler-cone membership plus metric positivity, and $\operatorname{Im} z_i$ need not
be positive componentwise. This is an intentional divergence from `is_physical`'s
last-resort test.

**The tadpole window is signed**, $0 < f\cdot\Sigma\cdot h \le Q$. Taking $|\cdot|$
first would admit *negative* D3 charge, which is unphysical.

**Stability goes through `classify_solution`**, which diagonalises the real-basis
$\partial^2 V$ — never `mass_matrix`, whose complex-basis branches have open defects
on this class of vacua.

### Putting it to work

Recall that our LCS record was built from the values NB05 *prints*. Ask the class
whether it is a physical vacuum:

In [ ]:

def show_report(vac, model, **kw):
    """Print a diagnostics report as a table."""
    report = vac.diagnostics(model, **kw)
    verdict = vac.is_consistent(model, **kw)
    print(f"is_consistent = {verdict}\n")
    for name, (ok, value, reason) in report.items():
        if str(reason).startswith("skipped"):
            tag, shown = "skip", ""
        else:
            tag, shown = ("ok" if ok else "FAIL"), (
                "" if value is None else f"{float(np.real(value)):+.4e}")
        note = f"   {reason}" if reason else ""
        print(f"  {name:20s} {tag:5s} {shown:>12s}{note}")
    return report


_ = show_report(vac_lcs_raw, finder, moduli_max=1e3)


The report does not merely say *no* — it says **which** constraint failed and by how
much: `max|DW| = 5.250e-07 exceeds residual_tol = 1.0e-08`. Every geometric condition
holds; the point is simply not converged to the default tolerance. That is a
diagnosis, and it is actionable.

So let us act on it: refine with the built-in Newton solver and re-check.

In [ ]:

z_ref, tau_ref, res_ref = finder.newton_method_flux_vacua(
    z_SUSY, tau_SUSY, flux_SUSY, mode="SUSY",
    tol=1e-14, max_iters=50, step_size_Newton=1.0,
)
print(f"Newton moved z by {float(jnp.max(jnp.abs(z_ref - z_SUSY))):.3e}, "
      f"tau by {abs(complex(tau_ref) - tau_SUSY):.3e}")
print(f"residual {vac_lcs_raw.residual:.3e}  ->  {float(res_ref):.3e}\n")

vac_lcs = as_vacuum(finder, z_ref, tau_ref, flux_SUSY)
_ = show_report(vac_lcs, finder, moduli_max=1e3)


The printed values were $2 \times 10^{-7}$ away from the true solution — enough to fail
an $10^{-8}$ F-term tolerance while looking perfectly reasonable to the eye. This is
precisely the class of mistake a bare `bool` hides and a named report surfaces.

For the rest of the notebook `vac_lcs` is the refined record. The coni-LCS vacuum was
solved to $10^{-13}$ from the start, so it needs no such repair:

In [ ]:

_ = show_report(vac_coni, model_coni, moduli_max=1e3)


Note `tadpole = +1.0400e+02` against $Q = 104$: this vacuum **saturates** the D3
tadpole bound exactly. It passes — the window is $0 < N_{\rm flux} \le Q$, inclusive
at the top — but it sits precisely on the boundary, which is the kind of detail worth
seeing in a report rather than inferring later.


## 5. Skipped is not passed

Some checks need an input that may not be available: the dilaton floor needs a
weak-coupling scale, the runaway bound needs a cutoff, the tadpole needs $Q$. When the
input is missing the check reports `reason` starting with `"skipped: "` and is
**excluded from the verdict**.

This is the one place where a validity checker can quietly lie. If a missing input
counted as a pass, a report from a model without a sampler would look *better* than
one with it. Here a skipped entry still carries `ok=True` so that naive
`all(...)` scans do not break, but `is_consistent` filters on the reason.

A bare `FluxEFT` has no sampler; the `FluxVacuaFinder` we have been using does
(`s_lower = 2.0`). Same vacuum, same code, different amount of evidence:

In [ ]:

eft_only = jvc.FluxEFT(h12=2, model_ID=1)          # no sampler
print(f"FluxEFT   has sampler: {getattr(eft_only, 'sampler', None) is not None}")
print(f"finder    has sampler: {getattr(finder, 'sampler', None) is not None}"
      f"   s_lower = {finder.sampler.s_lower}\n")

for label, model in (("bare FluxEFT", eft_only), ("FluxVacuaFinder", finder)):
    rep = vac_lcs.diagnostics(model, refresh=True)
    graded = [k for k, v in rep.items() if not str(v[2]).startswith("skipped")]
    skipped = {k: v[2] for k, v in rep.items() if str(v[2]).startswith("skipped")}
    print(f"{label:16s} graded {len(graded)}/{len(rep)}")
    for k, why in skipped.items():
        print(f"                   skip  {k:20s} {why}")
    print()


Supplying the inputs explicitly un-skips the checks — you do not need a sampler, just
a number:

In [ ]:

rep = vac_lcs.diagnostics(eft_only, s_min=1.0, moduli_max=1e3, refresh=True)
for k in ("dilaton_floor", "runaway_bound"):
    ok, value, reason = rep[k]
    print(f"  {k:16s} ok={ok}  value={float(np.real(value)):.4f}  "
          f"reason={reason or '(passed)'}")


## 6. Broken vacua name their own failure

The report earns its keep when something is wrong. Below, four deliberate defects —
each breaking exactly one condition — and in every case the failing key *and* a
readable reason.

The third case is worth reading closely. To produce **negative** D3 charge it is not
enough to flip the whole flux vector: $N_{\rm flux} = f \cdot \Sigma \cdot h$ is
*bilinear*, so flipping both $f$ and $h$ leaves it completely unchanged. Only the RR
block is flipped below, and the check then correctly rejects the record.

In [ ]:

x_good = np.asarray(vac_lcs.x, dtype=float)
f_good = np.asarray(vac_lcs.flux, dtype=float)
n_half = f_good.shape[0] // 2

# (1) axio-dilaton in the lower half-plane
x_bad_tau = x_good.copy()
x_bad_tau[-1] *= -1.0

# (2) a flux entry that is not an integer
f_frac = f_good.copy()
f_frac[0] += 0.25

# (3) negative D3 charge -- flip ONLY the RR block (bilinear: a full flip is a no-op)
f_neg = f_good.copy()
f_neg[:n_half] *= -1.0

cases = [
    ("Im(tau) < 0",        Vacuum(flux=f_good, x=x_bad_tau), {}),
    ("non-integral flux",  Vacuum(flux=f_frac, x=x_good),    {}),
    ("negative D3 charge", Vacuum(flux=f_neg,  x=x_good),    {}),
    ("runaway moduli",     Vacuum(flux=f_good, x=x_good),    {"moduli_max": 1.0}),
]

print(f"sanity: N_flux(full flip) = {float(np.real(finder.tadpole(-f_good))):+.1f} "
      f"vs original {float(np.real(finder.tadpole(f_good))):+.1f}  <- unchanged, bilinear")
print(f"        N_flux(RR flip)   = {float(np.real(finder.tadpole(f_neg))):+.1f}"
      f"  <- sign flipped\n")

for label, vac, kw in cases:
    rep = vac.diagnostics(finder, **kw)
    failing = {k: v for k, v in rep.items()
               if not str(v[2]).startswith("skipped") and not v[0]}
    print(f"{label:20s} is_consistent = {vac.is_consistent(finder, **kw)}")
    for k, (_ok, _val, reason) in failing.items():
        print(f"  {'':20s} FAIL {k:18s} {reason}")


Exactly one check fails in each case, and the reason carries the offending number. A
caller can branch on the key (`"tadpole"`), log the reason, or route the record to a
different pipeline — none of which is possible with a bare boolean.


## 7. The derived tier: `VacuumAnalysis`

Running the checks computes things worth keeping — eigenvalues of the Kähler metric,
and of the Hessian if you asked for stability. Those land in a single optional
sub-record:

```python
@dataclass
class VacuumAnalysis:
    checks:                    dict | None   # the report itself
    kahler_metric_eigenvalues: Array | None
    hessian_eigenvalues:       Array | None  # stability=True only
    mass_eigenvalues:          Array | None  # producer-set; see below
    alignment:                 float | None  # conifold + PFV
    args:                      dict | None   # the tolerances actually used
```

Grouping them serves a specific purpose: an entire tier can be dropped with **one
flag** when storing large ensembles (section 12), because everything here is
*reproducible* from the core record given a model. It is payload, not identity —
which is also why two vacua at the same point with the same flux compare equal
whether or not one of them carries a cached report.

`mass_eigenvalues` is **producer-set**: `diagnostics` never computes it. Stability is
checked through `classify_solution`, and the mass matrix is deliberately avoided.
Attach a spectrum yourself if you have one.

In [ ]:

rep = vac_lcs.diagnostics(finder, moduli_max=1e3, stability=True)
an = vac_lcs.analysis

print(f"analysis.summary()            : {an.summary()}")
print(f"tolerances used (args)        : {an.args}")
print(f"kahler_metric_eigenvalues     : {np.asarray(an.kahler_metric_eigenvalues)}")
print(f"hessian_eigenvalues           : {np.asarray(an.hessian_eigenvalues)}")
print(f"mass_eigenvalues              : {an.mass_eigenvalues}  (producer-set)")
print()
print(f"hessian_min_eig check         : {rep['hessian_min_eig'][0]}  "
      f"(min eig = {rep['hessian_min_eig'][1]:+.4e})")
print("-> all eigenvalues positive, so this is a genuine minimum of V, not a saddle.")


### Caching, and why the tolerances are stored

A second call with the same arguments reuses the report — the $O(h_{12}^3)$
eigendecomposition runs once. Change a tolerance and it recomputes, because
`analysis.args` no longer matches. That is what `args` is for.

In [ ]:

first = vac_lcs.diagnostics(finder, moduli_max=1e3, stability=True)
again = vac_lcs.diagnostics(finder, moduli_max=1e3, stability=True)
print(f"same object returned (cache hit): {first is again}")

tighter = vac_lcs.diagnostics(finder, moduli_max=1e3, residual_tol=1e-16)
print(f"tightened residual_tol -> recomputed: {tighter is not first}")
print(f"  residual now passes? {tighter['residual'][0]}  ({tighter['residual'][2]})")
print(f"  ... and `refresh=True` forces a recompute at identical tolerances too.")


### A hierarchy worth looking at

The coni-LCS Hessian shows why the check reports the *minimum eigenvalue* as its value
rather than only a boolean — and, if we also look at the eigen*vectors*, why the
reduced-EFT machinery of [NB12](12_freezer.ipynb) is legitimate here.

Slots $0,1$ of the real vector are $(\operatorname{Re} z_{\rm cf},
\operatorname{Im} z_{\rm cf})$, so the weight $\sum_{i\in\{0,1\}} v_i^2$ of an
eigenvector $v$ says how much of that direction lies in the conifold plane.

In [ ]:

vac_coni.diagnostics(model_coni, moduli_max=1e3, stability=True)
eigs = np.asarray(vac_coni.analysis.hessian_eigenvalues)

# Eigenvectors are not cached (only the spectrum is), so recompute to see WHICH
# direction each eigenvalue belongs to.
H = np.asarray(model_coni.ddV_x(np.asarray(vac_coni.x), np.asarray(vac_coni.flux)))
w, V = np.linalg.eigh(0.5 * (H + H.T))

print("coni-LCS Hessian: eigenvalue   weight in the z_cf plane")
for i, ev in enumerate(w):
    print(f"    {ev:+.6e}       {float(np.sum(V[:2, i] ** 2)):.4f}")
print(f"\nspectrum spans {w.max() / w.min():.2e}  "
      f"(~{np.log10(w.max() / w.min()):.0f} orders of magnitude)")
print(f"cached spectrum matches: {np.allclose(np.sort(eigs), w, rtol=1e-10)}")
print()
print("The heaviest pair carries weight 1.0000 and every other eigenvector 0.0000:")
print("at this point the Hessian block-diagonalises exactly into the conifold plane")
print("and the bulk.  The steep logarithmic period is what makes z_cf that stiff --")
print("and this separation is precisely what lets the ConifoldFreezer integrate it")
print("out and work with the light fields alone (NB12).")


## 8. The alignment scalar

For a PFV-style vacuum near the conifold there is a dimensionless diagnostic
measuring how close the configuration sits to the idealised AdS window
([arXiv:2009.03312](https://arxiv.org/abs/2009.03312)):

$$
\Xi \;=\; \frac{2\,\kappa\,V_{\tilde X}^{1/3}\,(2Q)\,|z_{\rm cf}|^{4/3}}
                {\left(g_s\,M_0\,|W_0|\right)^{2}},
\qquad \kappa = 89.5643 .
$$

It is **not** a storable field, and the reason is instructive: it needs the *model*
(for the mirror volume $V_{\tilde X}$ and the D3 cap $Q$), the location, **and** $M_0$
— the leading $M$-flux entry, i.e. PFV quantum numbers. A quantity requiring all
three cannot be filled in by whoever constructs the record. So it is computed on
demand into `analysis`, and only where it means anything:

In [ ]:

for name, vac, model in (("LCS", vac_lcs, finder),
                         ("coni-LCS", vac_coni, model_coni)):
    vac.diagnostics(model, refresh=True)
    xi = vac.analysis.alignment
    shown = "None" if xi is None else f"{xi:.4e}"
    print(f"  {name:9s} has_conifold={str(vac.has_conifold):5s}  alignment = {shown}")

print()
print("`conifold_alignment` is also callable directly, and is TOTAL: a missing Q or")
print("mirror volume returns None rather than raising or producing NaN.")
from jaxvacua.vacuum import conifold_alignment
xi_direct = conifold_alignment(model_coni, vac_coni.z, vac_coni.zcf,
                               vac_coni.gs, vac_coni.W0, float(M_coni[0]))
xi_nomodel = conifold_alignment(None, vac_coni.z, vac_coni.zcf,
                                vac_coni.gs, vac_coni.W0, float(M_coni[0]))
print(f"  direct call : {xi_direct:.4e}")
print(f"  no model    : {xi_nomodel}")


## 9. The `PFV` subclass

`PFV` is a `Vacuum` that additionally carries the perturbatively-flat-vacuum algebra
in a composed `PFVData`: the quanta $(\vec M, \vec K)$, the derived $N$-matrix, the
flat direction $\vec p = N^{-1}K$, the full flux vector, and the PFV conditions. It
also exposes the analytic 2-term-racetrack **estimate**, kept deliberately separate
from the solved fields.

In [ ]:

# The LCS PFV of NB10: same geometry as NB10's `model_LCS`.
model_LCS = jvc.FluxVacuaFinder(h12=2, model_ID=1, maximum_degree=2,
                                limit="LCS", model_type="KS")
M_pfv = jnp.array([-16.0, 50.0])
K_pfv = jnp.array([3.0, -4.0])
rt = model_LCS.pfv_racetrack(M_pfv, K_pfv)

pfv = PFV.from_quantum_numbers(model_LCS, M_pfv, K_pfv, complex(rt["tau0"]))
print(f"data           : {pfv.data}")
print(f"flat direction p = {np.asarray(pfv.data.p)}")
print(f"tau_input      : {pfv.tau_input:.6f}")
print()
print("racetrack ESTIMATE (seed), distinct from the solved fields:")
print(f"  tau0             = {complex(pfv.tau0):.6f}")
print(f"  gs_estimate      = {float(pfv.gs_estimate):.4f}")
print(f"  log10|W0|_est    = {float(pfv.log10_W0_estimate):.3f}")
print(f"  solved W0 field  = {pfv.W0}   <- NaN: this seed has not been refined")


### A `PFV` is checked against actually *being* one

`PFV.diagnostics` extends the base report with the PFV algebra conditions
(arXiv:2512.17095 Eq. 6.27/7.28) under `pfv:`-prefixed keys. The prefix is not
decoration: the PFV set has its own tadpole condition on $-M\cdot K/2$, which is a
*different quantity* from the base `tadpole` check on $f\cdot\Sigma\cdot h$, and the
two must not collide.

Consequently `is_consistent` on a `PFV` covers the algebra automatically: a record
labelled a PFV whose $\det N = 0$, or whose $\vec p$ leaves the Kähler cone, reports
`False` with the offending condition named.

In [ ]:

rep = pfv.diagnostics(model_LCS)
print("PFV algebra conditions:")
for k, (ok, value, reason) in rep.items():
    if k.startswith("pfv:"):
        val = np.asarray(value)
        shown = f"{val.item():+.6g}" if val.size == 1 else str(val.tolist())
        print(f"  {k[4:]:24s} {'ok' if ok else 'FAIL':4s}  {shown}")
print()
print("(the 'p' entry is filtered out: it carries the p-vector, not a verdict)")

In [ ]:

# A flux that does NOT satisfy the conditions: reported, not silently accepted.
M_bad = jnp.array([-6.0, -6.0])
pfv_bad = PFV.from_quantum_numbers(model_LCS, M_bad, K_pfv, complex(0.0, 6.85))
rep_bad = pfv_bad.diagnostics(model_LCS)
failing = [k for k, v in rep_bad.items()
           if k.startswith("pfv:") and not v[0]]
print(f"M = {np.asarray(M_bad).astype(int).tolist()}, "
      f"K = {np.asarray(K_pfv).astype(int).tolist()}")
print(f"  is_consistent = {pfv_bad.is_consistent(model_LCS)}")
for k in failing:
    print(f"  FAIL {k}: {rep_bad[k][2]}")

In [ ]:

# A singular N-matrix has no flat direction p = N^-1 K, hence no location to seed
# from -- so construction raises rather than returning an unusable object.
try:
    PFV.from_quantum_numbers(model_LCS, jnp.array([-4.0, 0.0]), K_pfv,
                             complex(0.0, 6.85))
except ValueError as exc:
    print(f"ValueError: {exc}")

# The algebra layer alone is deliberately more permissive: it reports p = None.
pd = PFVData.from_fluxes(model_LCS, jnp.array([-4.0, 0.0]), K_pfv)
print(f"\nPFVData tolerates it: p = {pd.p},  det N != 0 -> "
      f"{bool(np.asarray(pd.check()['det N!=0'][0]))}")


## 10. Three notions of sameness

"Are these the same vacuum?" has three useful answers, and picking the wrong one is a
common source of confusion.

| call | identifies | needs a model |
|---|---|---|
| `a.equals(b)` | *records*, every field, **bit-exactly** | no |
| `a.equivalent_to(b)` | the same **point**, rounded | no |
| `a.equivalent_to(b, finder)` | points related by **duality** | yes |

`equals` is exact **on purpose**: its job is to prove that a serialisation round trip
or a parity re-run lost *nothing*, and a tolerance would silently defeat that. It is
therefore the wrong tool for comparing two independent solves, whose last digits
always differ.

`equivalent_to` rounds — moduli and $\tau$ to `n_digits` decimals (6 by default) and
the flux to integers, via `flux_utils.dedup_key`, whose whole purpose is "to absorb
numerical noise from solver convergence".

In [ ]:

# The same vacuum, perturbed at the level of solver noise.
x_noise = np.asarray(vac_lcs.x, dtype=float).copy()
x_noise[0] += 1e-12
vac_noise = Vacuum(flux=np.asarray(vac_lcs.flux), x=x_noise, limit="LCS")

x_far = np.asarray(vac_lcs.x, dtype=float).copy()
x_far[0] += 1e-3
vac_far = Vacuum(flux=np.asarray(vac_lcs.flux), x=x_far, limit="LCS")

print(f"perturbed by 1e-12:  equals = {vac_lcs.equals(vac_noise)},  "
      f"equivalent_to = {vac_lcs.equivalent_to(vac_noise)}")
print(f"perturbed by 1e-03:  equals = {vac_lcs.equals(vac_far)},  "
      f"equivalent_to = {vac_lcs.equivalent_to(vac_far)}")
print()
# the rounding is a knob
x_mid = np.asarray(vac_lcs.x, dtype=float).copy()
x_mid[0] += 1e-4
vac_mid = Vacuum(flux=np.asarray(vac_lcs.flux), x=x_mid, limit="LCS")
print(f"perturbed by 1e-04:  n_digits=6 -> {vac_lcs.equivalent_to(vac_mid)},   "
      f"n_digits=2 -> {vac_lcs.equivalent_to(vac_mid, n_digits=2)}")


Note also that `equivalent_to` **ignores the subclass**. A base `Vacuum` re-solved
from scratch and a `PFV` sitting at the same point with the same flux *are* the same
vacuum, even though they are not the same record:

In [ ]:

pfv_same_point = PFV(flux=np.asarray(vac_lcs.flux), x=np.asarray(vac_lcs.x),
                     tau_input=complex(vac_lcs.tau))
print(f"Vacuum vs PFV at the same point:  equals = {vac_lcs.equals(pfv_same_point)},  "
      f"equivalent_to = {vac_lcs.equivalent_to(pfv_same_point)}")


### Up to duality

With a model, the comparison additionally quotients by the
$SL(2,\mathbb{Z}) \times$ monodromy action: both points are mapped into the
fundamental domain first. Because that action moves $\tau$ **and** the flux together,
a duality pair looks like two genuinely different records that describe one physical
vacuum.

Below we take the solved vacuum off the fundamental domain ($\tau \to \tau + 1$) and
ask `map_to_fd` for its representative — which comes back with the correspondingly
transformed flux.

In [ ]:

from jaxvacua.flux_utils import map_to_fd

tau_off = complex(vac_lcs.tau) + 1.0
z_fd, tau_fd, flux_fd = map_to_fd(finder, jnp.asarray(vac_lcs.z), tau_off,
                                  jnp.asarray(vac_lcs.flux), enabled=True)

vac_off = Vacuum(flux=np.asarray(vac_lcs.flux), z=vac_lcs.z, tau=tau_off)
vac_img = Vacuum(flux=np.asarray(flux_fd, dtype=float), z=np.asarray(z_fd),
                 tau=complex(tau_fd))

print(f"off the FD : tau = {tau_off:.6f}   flux = "
      f"{np.asarray(vac_off.flux).astype(int).tolist()}")
print(f"FD image   : tau = {complex(tau_fd):.6f}   flux = "
      f"{np.asarray(vac_img.flux).astype(int).tolist()}")
print("             ^ different tau AND different flux: SL(2,Z) acts on both")
print()
print(f"  equals                      = {vac_off.equals(vac_img)}")
print(f"  equivalent_to (model-free)  = {vac_off.equivalent_to(vac_img)}"
      "    <- different points, correctly")
print(f"  equivalent_to (with finder) = {vac_off.equivalent_to(vac_img, finder)}"
      "     <- one vacuum, two frames")


## 11. Deduplicating an ensemble

`unique_vacua` is the bulk form of the same idea: a single $O(n)$ pass over the
hashable key, preserving insertion order. Pass a model to collapse duality orbits;
omit it to collapse only coincident points — which is what a finder-free consumer,
such as a vault read-back, can do.

In [ ]:

ensemble = [vac_lcs, vac_noise, vac_off, vac_img, vac_far]
print(f"ensemble of {len(ensemble)}: refined, noise-perturbed(1e-12), off-FD, "
      "FD image, far(1e-3)")
print()
print(f"  unique_vacua(...)          -> {len(unique_vacua(ensemble))} representatives"
      "   (coincident points merged)")
print(f"  unique_vacua(..., finder)  -> {len(unique_vacua(ensemble, finder))} "
      "representatives   (+ duality orbits merged)")
print()
print("`dedup_vacua` is an alias of `unique_vacua`.")


## 12. Storage: three tiers

**Never pickle a `Vacuum` instance.** `to_dict` / `from_dict` is the canonical
interchange, and both shipped writers go through it, so a stored file survives a
refactor of these classes — which matters, because these classes *have* been
refactored.

| use | format | why |
|---|---|---|
| local / dev | `save_vacua` — gzip + pickled **dicts** | convenient, exact dtypes; the payload is dicts of arrays, not class instances |
| **public / vault** | `vacuum_to_json` — tagged JSON | no unpickle-on-load, so no remote-code-execution vector for data others download; human-inspectable and diffable |
| large ensembles | columnar (StringForge's writer) over the **minimal** tier | `analysis=False` makes a row small and fully queryable |

The tagged JSON round-trips NumPy arrays and complex scalars losslessly via two tags,
`{"__nd__": …, "dtype": …}` and `{"__c__": [re, im]}`.

In [ ]:

tmp = tempfile.mkdtemp()

# --- tier 1: local, gzip + pickled dicts ---------------------------------
path = os.path.join(tmp, "vacua.p")
save_vacua([vac_lcs, vac_coni, pfv], path)
back = load_vacua(path)
print(f"save/load: {[type(v).__name__ for v in back]}")
print(f"  faithful: {all(a.equals(b) for a, b in zip([vac_lcs, vac_coni, pfv], back))}")

# --- tier 2: download-facing, tagged JSON --------------------------------
doc = vacuum_to_json(vac_coni)
rebuilt = vacuum_from_json(doc)
print(f"\nJSON: {len(doc)} chars, rebuilt as {type(rebuilt).__name__}, "
      f"faithful: {vac_coni.equals(rebuilt)}")
print(f"  arrays and complex survive: x exact = "
      f"{np.array_equal(np.asarray(rebuilt.x), np.asarray(vac_coni.x))}, "
      f"W0 exact = {rebuilt.W0 == vac_coni.W0}")

# --- the tier switch -----------------------------------------------------
full = vacuum_to_json(vac_coni, analysis=True)
minimal = vacuum_to_json(vac_coni, analysis=False)
print(f"\nanalysis=True  {len(full):6d} chars")
print(f"analysis=False {len(minimal):6d} chars   "
      f"({100 * (1 - len(minimal) / len(full)):.0f}% smaller)")
print(f"  and the core record is unchanged: "
      f"{vac_coni.equals(vacuum_from_json(minimal))}")
print("  -> `analysis` is payload, not identity: dropping it does not change")
print("     WHICH vacuum this is, only how much precomputation travels with it.")


### The subclass registry — and what is *not* in this class

`to_dict` stamps a `_kind` discriminator and a `_schema_version`; `from_dict` looks
`_kind` up in a registry. Downstream packages register their own types with
`register_vacuum_kind`, so their records rebuild faithfully through the plain
`jaxvacua` loader. An *unregistered* kind degrades to a base `Vacuum` with a
`RuntimeWarning` rather than failing the load.

This is how the promotion layer lives outside `jaxvacua`. The private `afvs` package
defines `AFV` (an ISD-proxy seed) and `PromotedPFV`, which carry what a plain solved
vacuum has no notion of:

* `genealogy` — which promotion steps ran,
* `success` and `is_solved()` — did the final step converge,
* `trajectory` — one row per step, with its own rendered table.

**None of that belongs on a `Vacuum`.** A vacuum from a Newton solve has no promotion
history, so "did the final step converge?" is not a question about it — which is why
this notebook never needed those fields. Where a validity statement is wanted, the
diagnostics report is the answer, and unlike a producer-set flag it can be
*recomputed and checked*.

In [ ]:

d = vac_coni.to_dict()
print(f"_kind = {d['_kind']!r}   _schema_version = {d['_schema_version']}")
print(f"derived z/tau are NOT stored (re-derived from x): "
      f"{'z' not in d and 'tau' not in d}")
print()

# an unregistered kind is a warning, not a crash
d_unknown = dict(d, _kind="SomePackageVacuum")
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    degraded = Vacuum.from_dict(d_unknown)
print(f"unknown kind -> {type(degraded).__name__}, warned: "
      f"{any('Unknown vacuum kind' in str(w.message) for w in caught)}")
print()
print(f"base Vacuum has provenance fields? "
      f"{any(hasattr(vac_lcs, a) for a in ('genealogy', 'success', 'trajectory'))}")


## 13. Feature reference

**Core record** — `flux` (required) · `x` or `(z, tau)` (one required, the other
derived) · `limit` · `W0` · `DW` · `residual` · `gs` · `metadata`

**Conifold group**, meaningful iff `has_conifold` — `zcf` · `residual_bulk` ·
`residual_conifold`

**Validity**

| | |
|---|---|
| `diagnostics(model, *, moduli_max, s_min, residual_tol, stability, refresh)` | the named report `{check: (ok, value, reason)}`, cached into `analysis` |
| `is_consistent(model, **kw)` | its reduction over the non-skipped checks |
| `has_conifold` | whether the conifold group applies |

**Derived** — `analysis` (`VacuumAnalysis`): `checks` · `kahler_metric_eigenvalues` ·
`hessian_eigenvalues` · `mass_eigenvalues` (producer-set) · `alignment` · `args`

**Comparison** — `equals` (exact) · `equivalent_to(other[, finder], n_digits=6)`
(tolerant, optionally up to duality) · `canonical_key([finder], n_digits=6)`

**Serialisation** — `to_dict(analysis=True)` / `from_dict` · `save_vacua` /
`load_vacua` (gzip+pickled dicts) · `vacuum_to_json` / `vacuum_from_json` (tagged
JSON) · `register_vacuum_kind`

**Display** — `short_summary()` · `long_summary()`

**Module helpers** — `real_to_complex` / `complex_to_real` · `unique_vacua` /
`dedup_vacua` · `conifold_alignment` · `resolve_hyperplanes` · `encode_json` /
`decode_json`

**`PFV` adds** — `data` (`PFVData`) · `tau_input` · `from_quantum_numbers` ·
`racetrack_estimate()` · `tau0` · `W0_estimate` · `log10_W0_estimate` ·
`gs_estimate` · the `pfv:` condition checks

## Take-aways

* A `Vacuum` is **flux + location**. Both are mandatory; the two coordinate views are
  kept consistent for you, and malformed input is rejected at construction rather
  than surfacing later as a `None`.
* **`limit` gates the conifold data.** An LCS vacuum has no $z_{\rm cf}$ and no
  residual split — those are undefined, not unknown, and read as `None` rather than
  `NaN`.
* **`diagnostics(model)` explains; `is_consistent(model)` decides**, and the two
  cannot disagree because the second is a reduction of the first. A check whose
  inputs are missing is *skipped* and excluded — never silently passed.
* Section 4 was the point of the exercise: values that look perfectly reasonable were
  $2\times10^{-7}$ off, and the report named the failing check and the number.
* **Derived data lives in `analysis`** and drops with one flag, so the same class
  serves a bulk ensemble and a published dataset.
* **`equals` is exact, `equivalent_to` is tolerant, and with a model it is
  duality-aware.** Use the second for solver output.
* **Provenance is not here.** `genealogy` / `success` / `trajectory` belong to the
  promotion layer in `afvs`, and a producer-set flag is a weaker statement than a
  report you can recompute.

## Further reading

* [NB05 — Finding flux vacua](../02_vacuum_finding/05_finding_flux_vacua.ipynb) — where the LCS solution comes from
* [NB10 — coni-LCS pipeline](../03_geometry_and_limits/10_coniLCS_pipeline.ipynb) — the coni-LCS vacuum and the PFV construction
* [NB12 — Freezer](12_freezer.ipynb) — integrating out the stiff conifold direction seen in section 7
* [NB14 — Hessian analysis](14_hessian_analysis.ipynb) — the mass spectrum in more depth
* [NB15 — Landscape statistics](15_landscape_statistics.ipynb) — ensembles of vacua
* Demirtas, Kim, McAllister, Moritz, *Conifold Vacua with Small Flux Superpotential*, [arXiv:2009.03312](https://arxiv.org/abs/2009.03312)
* JAXVacua framework paper, [arXiv:2306.06160](https://arxiv.org/abs/2306.06160)